# Stage 1: data, validator, executor

What this notebook shows:
1. the synthetic legacy production DB (tables, row counts, code columns),
2. the synthetic question sets (40 test questions in PL and EN, 400 train pairs, 40 val pairs),
3. the SQL validator and the read-only executor,
4. the result comparison used for execution accuracy,
5. BIRD status (download + stratified 150-question sample).

Everything here is computed from files produced by `uv run t2sbench build-synthetic`; rerun the notebook to refresh.

In [1]:
import json, sqlite3
from pathlib import Path
import pandas as pd
from t2sbench.datasets.synthetic.build import build_all, read_jsonl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SYN = ROOT / "data" / "synthetic"
if not (SYN / "prod.sqlite").exists():
    build_all(SYN, seed=42)
manifest = json.loads((SYN / "manifest.json").read_text())
con = sqlite3.connect(f"file:{SYN / 'prod.sqlite'}?mode=ro", uri=True)
pd.Series(manifest["row_counts"], name="rows").to_frame().T

,PRZYCZ,BR_TYP,LIN,MCH,ZMIANA,PROD,OPER,ZLEC,ZLEC_POZ,PROD_REJ,BRAKI,OPER_ZM,PRZEST,MCH_STAT
rows,11,10,7,22,270,40,66,420,1691,3409,2283,3409,1527,3054


## Schema (legacy naming)

In [2]:
for (sql,) in con.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name IN ('ZLEC','PRZEST','PROD_REJ')"):
    print(sql, end="\n\n")

CREATE TABLE PRZEST (
    PRZ_ID     INTEGER PRIMARY KEY,
    MCH_ID     INTEGER NOT NULL REFERENCES MCH(MCH_ID),
    ZM_ID      INTEGER NOT NULL REFERENCES ZMIANA(ZM_ID),
    TS_OD      TEXT NOT NULL,
    TS_DO      TEXT NOT NULL,
    CZAS_MIN   INTEGER NOT NULL,
    PRZYCZ_KOD TEXT NOT NULL REFERENCES PRZYCZ(PRZYCZ_KOD),
    PLAN_FL    TEXT NOT NULL,
    UWAGI      TEXT
)

CREATE TABLE ZLEC (
    ZLEC_ID    INTEGER PRIMARY KEY,
    ZLEC_NR    TEXT NOT NULL,
    PROD_ID    INTEGER NOT NULL REFERENCES PROD(PROD_ID),
    ZLEC_ILOSC INTEGER NOT NULL,
    ZLEC_STAT  TEXT NOT NULL,
    PRIOR      TEXT NOT NULL,
    DT_UTW     TEXT NOT NULL,
    DT_PLAN_OD TEXT NOT NULL,
    DT_PLAN_DO TEXT NOT NULL
)

CREATE TABLE PROD_REJ (
    REJ_ID     INTEGER PRIMARY KEY,
    ZM_ID      INTEGER NOT NULL REFERENCES ZMIANA(ZM_ID),
    MCH_ID     INTEGER NOT NULL REFERENCES MCH(MCH_ID),
    ZLEC_ID    INTEGER NOT NULL REFERENCES ZLEC(ZLEC_ID),
    POZ_ID     INTEGER NOT NULL REFERENCES ZLEC_POZ(POZ_ID),


## Code columns: values in the DB and their meaning from `codes.yaml`

In [3]:
from t2sbench.datasets.synthetic.generator import load_codes
codes = load_codes()
rows = []
for col in ["ZLEC.ZLEC_STAT", "PRZEST.PLAN_FL", "PRZYCZ.KAT", "BRAKI.BR_STAT", "ZMIANA.ZM_KOD"]:
    t, c = col.split(".")
    for v, n in con.execute(f"SELECT {c}, COUNT(*) FROM {t} GROUP BY {c} ORDER BY {c}"):
        rows.append({"column": col, "code": v, "rows": n, "meaning_en": codes[col][str(v)]["en"], "meaning_pl": codes[col][str(v)]["pl"]})
pd.DataFrame(rows)

,column,code,rows,meaning_en,meaning_pl
0,ZLEC.ZLEC_STAT,A,33,cancelled,anulowane
1,ZLEC.ZLEC_STAT,N,93,new / not started,nowe / nierozpoczęte
2,ZLEC.ZLEC_STAT,R,49,in progress,w realizacji
3,ZLEC.ZLEC_STAT,W,4,on hold,wstrzymane
4,ZLEC.ZLEC_STAT,Z,241,completed / closed,zakończone / zamknięte
5,PRZEST.PLAN_FL,N,915,unplanned downtime,przestój nieplanowany
6,PRZEST.PLAN_FL,T,612,planned downtime,przestój planowany
7,PRZYCZ.KAT,J,1,quality-related,jakościowa
8,PRZYCZ.KAT,O,3,organisational,organizacyjna
9,PRZYCZ.KAT,P,3,planned,planowa


## Test set: one PL/EN pair per template, with the gold result

In [4]:
test = read_jsonl(SYN / "test.jsonl")
pd.set_option("display.max_colwidth", 140)
out = []
for r in test:
    if r["pair_id"].endswith("-0"):
        res = con.execute(r["sql"]).fetchall()
        out.append({"template": r["template_id"], "lang": r["lang"], "question": r["question"], "gold_result": str(res[:3])[:90]})
pd.DataFrame(out)

,template,lang,question,gold_result
0,T01,pl,Ile dobrych sztuk wyprodukowała linia L06 na zmianie popołudniowej 10 marca 2026?,"[(237,)]"
1,T01,en,"How many good units did line L06 produce on the afternoon shift on March 10, 2026?","[(237,)]"
2,T02,pl,"Pokaż 3 najdłuższe przestoje na linii L06 w marcu 2026: kod maszyny, opis przyczyny i czas w minutach.","[('M0603', 'AWARIA HYDRAULICZNA', 264), ('M0603', 'AWARIA HYDRAULICZNA', 263), ('M0601', '"
3,T02,en,"Show the 3 longest downtimes on line L06 in March 2026: machine code, reason description and duration in minutes.","[('M0603', 'AWARIA HYDRAULICZNA', 264), ('M0603', 'AWARIA HYDRAULICZNA', 263), ('M0601', '"
4,T03,pl,Jaki jest procent braków dla zlecenia ZP/2026/00415?,"[(1.8618324350808426,)]"
5,T03,en,What is the scrap rate in percent for order ZP/2026/00415?,"[(1.8618324350808426,)]"
6,T04,pl,Ile razy maszyna P0104 uległa awarii w lutym 2026?,"[(5,)]"
7,T04,en,How many times did machine P0104 break down in February 2026?,"[(5,)]"
8,T05,pl,"Podaj liczbę nowych, jeszcze nierozpoczętych zleceń dotyczących grupy wyrobów „wsporniki i mocowania”.","[(21,)]"
9,T05,en,Give the number of new orders that have not started yet for the 'brackets and mounts' product group.,"[(21,)]"


## Train / val pools: templates are disjoint from the test templates

In [5]:
train, val = read_jsonl(SYN / "train.jsonl"), read_jsonl(SYN / "val.jsonl")
df = pd.DataFrame(train + val + test)
print("test templates :", sorted(df[df.split == "test"].template_id.unique()))
print("train templates:", sorted(df[df.split != "test"].template_id.unique()))
pd.crosstab(df.template_id, [df.split, df.lang]).T

test templates : ['T01', 'T02', 'T03', 'T04', 'T05', 'T06', 'T07', 'T08', 'T09', 'T10']
train templates: ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10', 'R11', 'R12', 'R13', 'R14', 'R15', 'R16', 'R17', 'R18', 'R19', 'R20']


template_id  R01  R02  R03  R04  R05  R06  R07  R08  R09  R10  ...  T01  T02  \
split lang                                                     ...             
test  en       0    0    0    0    0    0    0    0    0    0  ...    4    4   
      pl       0    0    0    0    0    0    0    0    0    0  ...    4    4   
train en      13   12   13    8    9   11   10    7    4    8  ...    0    0   
      pl      12   10   12    9    9   13   12    7    4    8  ...    0    0   
val   en       0    1    0    1    0    2    3    1    2    0  ...    0    0   
      pl       1    3    1    0    0    0    1    1    2    1  ...    0    0   

template_id  T03  T04  T05  T06  T07  T08  T09  T10  
split lang                                           
test  en       4    4    4    4    4    4    4    4  
      pl       4    4    4    4    4    4    4    4  
train en       0    0    0    0    0    0    0    0  
      pl       0    0    0    0    0    0    0    0  
val   en       0    0    0    0    0    0    0    0  
      pl       0    0    0    0    0    0    0    0  

[6 rows x 30 columns]

## Validator and read-only executor

In [6]:
from t2sbench.executor import execute
db = SYN / "prod.sqlite"
probes = [
    "SELECT COUNT(*) FROM ZLEC WHERE ZLEC_STAT = 'R'",
    "WITH x AS (SELECT LIN_KOD FROM LIN) SELECT * FROM x LIMIT 2",
    "DELETE FROM ZLEC",
    "SELECT 1; DROP TABLE LIN",
    "PRAGMA query_only = OFF",
    "ATTACH DATABASE 'other.db' AS o",
    "SELECT * FROM PROD_REJ",
    "WITH RECURSIVE c(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM c) SELECT COUNT(*) FROM c",
]
rows = []
for q in probes:
    r = execute(db, q, timeout_s=2)
    rows.append({"sql": q, "ok": r.ok, "error_kind": r.error_kind, "error": r.error, "n_rows": len(r.rows), "truncated": r.truncated, "seconds": round(r.elapsed_s, 2)})
pd.DataFrame(rows)

,sql,ok,error_kind,error,n_rows,truncated,seconds
0,SELECT COUNT(*) FROM ZLEC WHERE ZLEC_STAT = 'R',True,NaN,NaN,1,False,0.01
1,WITH x AS (SELECT LIN_KOD FROM LIN) SELECT * FROM x LIMIT 2,True,NaN,NaN,2,False,0.00
2,DELETE FROM ZLEC,False,validation,"only SELECT/WITH queries are allowed, got Delete",0,False,0.00
3,SELECT 1; DROP TABLE LIN,False,validation,"expected exactly one statement, got 2",0,False,0.00
4,PRAGMA query_only = OFF,False,validation,"only SELECT/WITH queries are allowed, got Pragma",0,False,0.00
5,ATTACH DATABASE 'other.db' AS o,False,validation,"only SELECT/WITH queries are allowed, got Attach",0,False,0.00
6,SELECT * FROM PROD_REJ,True,NaN,NaN,1000,True,0.00
7,WITH RECURSIVE c(n) AS (SELECT 1 UNION ALL SELECT n + 1 FROM c) SELECT COUNT(*) FROM c,False,timeout,timeout after 2s,0,False,2.00


## Result comparison (execution accuracy)

Multiset of tuples, numbers rounded to 4 decimals, row order ignored unless the gold SQL has a top-level `ORDER BY`.

In [7]:
from t2sbench.evaluate import execution_match
gold = test[0]["sql"]
cands = {
    "same query": gold,
    "different alias style": gold.replace("SUM(r.ILOSC_OK)", "SUM(r.ILOSC_OK) AS total"),
    "wrong shift": gold.replace("z.ZM_KOD = '", "z.ZM_KOD <> '"),
    "no answer": None,
}
pd.DataFrame([{"candidate": k, "correct": execution_match(db, v, gold).correct} for k, v in cands.items()])

,candidate,correct
0,same query,True
1,different alias style,True
2,wrong shift,False
3,no answer,False


## BIRD

In [8]:
sample = ROOT / "data" / "bird_dev_sample150.json"
if sample.exists():
    s = json.loads(sample.read_text())
    print({k: s[k] for k in ("n", "pool_size", "by_difficulty")}, "excluded:", len(s["excluded"]))
else:
    print("BIRD not downloaded in this environment. Run:\n  uv run t2sbench fetch-bird dev\n  uv run t2sbench sample-bird")

BIRD not downloaded in this environment. Run:
  uv run t2sbench fetch-bird dev
  uv run t2sbench sample-bird
